# スケーリング則

LLM の事前学習では、モデルサイズ、学習トークン数、計算量を増やすと損失が規則的に下がる領域があります。スケーリング則は、その観測を使って次の実験規模や予算配分を決めるための道具です。重要なのは式を暗記することではなく、観測点から指数を推定し、外挿の危うさを理解し、同じ計算量の中でモデルとデータをどう配分するかを判断することです。

## べき乗則で損失低下を近似する

よく使う形は L(N, D) = L_inf + a N^{-alpha} + b D^{-beta} です。N はパラメータ数、D は学習トークン数、L_inf は近い範囲では残りそうな損失の床です。alpha が大きいほどモデルサイズの増加が効きやすく、beta が大きいほどデータ追加が効きやすいと読めます。

In [ ]:
import math
import random
from statistics import mean

random.seed(19)

N_M = [30, 60, 120, 240, 480, 960]
D_B = [5, 10, 20, 40, 80, 160]

TRUE = {
    'floor': 1.58,
    'a': 3.35,
    'alpha': 0.37,
    'b': 2.10,
    'beta': 0.29,
}


def noisy(value, scale=0.008):
    return value + random.gauss(0.0, scale)

loss_by_N = [noisy(TRUE['floor'] + TRUE['a'] * (n ** -TRUE['alpha'])) for n in N_M]
loss_by_D = [noisy(TRUE['floor'] + TRUE['b'] * (d ** -TRUE['beta'])) for d in D_B]

print('model-size sweep')
for n, loss in zip(N_M, loss_by_N):
    print(f'N={n:>4}M loss={loss:.4f}')
print('data-size sweep')
for d, loss in zip(D_B, loss_by_D):
    print(f'D={d:>4}B loss={loss:.4f}')

## 床を仮置きして指数を読む

L_inf が分かっていれば、log(L - L_inf) と log N はほぼ直線になります。実際には床も未知なので、床の候補を走査し、その候補のもとで直線近似の誤差が小さいものを選びます。床の推定がずれると指数も動くため、推定値を固定の真実として扱いません。

In [ ]:
def linear_fit(xs, ys):
    mx = mean(xs)
    my = mean(ys)
    varx = mean((x - mx) ** 2 for x in xs)
    cov = mean((x - mx) * (y - my) for x, y in zip(xs, ys))
    slope = 0.0 if varx == 0 else cov / varx
    intercept = my - slope * mx
    return slope, intercept


def fit_power(xs, losses, floor):
    shifted = [y - floor for y in losses]
    if any(v <= 0 for v in shifted):
        return None
    lx = [math.log(x) for x in xs]
    ly = [math.log(v) for v in shifted]
    slope, intercept = linear_fit(lx, ly)
    exponent = -slope
    coef = math.exp(intercept)
    pred = [floor + coef * (x ** -exponent) for x in xs]
    mse = mean((p - y) ** 2 for p, y in zip(pred, losses))
    return {'floor': floor, 'coef': coef, 'exponent': exponent, 'mse': mse, 'pred': pred}

min_loss = min(loss_by_N + loss_by_D)
best = None
for i in range(500):
    floor = min_loss - 0.35 + i * 0.349 / 499
    fit_n = fit_power(N_M, loss_by_N, floor)
    fit_d = fit_power(D_B, loss_by_D, floor)
    if not fit_n or not fit_d:
        continue
    total = fit_n['mse'] + fit_d['mse']
    if best is None or total < best['mse']:
        best = {'floor': floor, 'N': fit_n, 'D': fit_d, 'mse': total}

print('estimated floor:', round(best['floor'], 4))
print('alpha:', round(best['N']['exponent'], 4), 'beta:', round(best['D']['exponent'], 4))
print('mse:', round(best['mse'], 8))

## 当てはまりと外挿を分ける

観測範囲でよく合う式でも、範囲外の予測は不安定です。小規模な観測点から大規模学習の損失を予測するときは、床、指数、データ品質、optimizer の違いがすべて外挿誤差になります。

In [ ]:
def predict_power(x, fit):
    return fit['floor'] + fit['coef'] * (x ** -fit['exponent'])

print('fit residuals for N sweep')
for n, obs, pred in zip(N_M, loss_by_N, best['N']['pred']):
    print(f'N={n:>4}M obs={obs:.4f} pred={pred:.4f} residual={obs-pred:+.4f}')

future_N = [1920, 3840, 7680]
print('extrapolation')
for n in future_N:
    print(f'N={n:>5}M predicted_loss={predict_power(n, best["N"]):.4f}')

## 同じ計算量で配分を選ぶ

デコーダ型モデルの粗い近似では、訓練計算量 C は 6ND に比例します。C を固定すると、モデルを大きくすれば読めるトークン数は減ります。反対に、トークン数を増やせばモデルは小さくなります。isoflops は、この同じ計算量の線上で損失が小さくなる N と D を探す考え方です。

In [ ]:
floor = best['floor']
alpha = best['N']['exponent']
beta = best['D']['exponent']
a_raw = best['N']['coef'] * (1e6 ** alpha)
b_raw = best['D']['coef'] * (1e9 ** beta)


def loss_nd(N_params, D_tokens):
    return floor + a_raw * (N_params ** -alpha) + b_raw * (D_tokens ** -beta)


def optimal_for_compute(C):
    numerator = a_raw * alpha
    denominator = b_raw * beta
    N = (numerator / denominator) ** (1 / (alpha + beta)) * ((C / 6.0) ** (beta / (alpha + beta)))
    D = C / (6.0 * N)
    return N, D

for C in [1e18, 3e18, 1e19, 3e19, 1e20]:
    n, d = optimal_for_compute(C)
    print(f'C={C:.1e} -> N={n/1e6:8.2f}M D={d/1e9:8.2f}B loss={loss_nd(n,d):.4f}')

## 配分ミスを数値で見る

同じ C でも、最適比よりモデルを大きくしすぎるとデータ不足になり、モデルを小さくしすぎると容量不足になります。損失差を比べると、予算配分の失敗がどちら側に出ているかを読めます。

In [ ]:
C = 1e20
n_opt, d_opt = optimal_for_compute(C)
print('reference C:', f'{C:.1e}')
print('optimal:', f'N={n_opt/1e6:.2f}M', f'D={d_opt/1e9:.2f}B', f'loss={loss_nd(n_opt,d_opt):.4f}')
for ratio in [0.25, 0.5, 1.0, 2.0, 4.0]:
    n = n_opt * ratio
    d = C / (6.0 * n)
    loss = loss_nd(n, d)
    side = 'data-heavy' if ratio < 1 else ('balanced' if ratio == 1 else 'model-heavy')
    print(f'{side:11s} N x {ratio:<4} -> N={n/1e6:8.2f}M D={d/1e9:8.2f}B loss={loss:.4f}')

## 小規模スイープの不確かさを測る

観測点が少ないと、指数推定は揺れます。ブートストラップで観測点を再標本化し、alpha と beta のばらつきを見ます。分散が大きいなら、大規模外挿の前に追加実験が必要です。

In [ ]:
def bootstrap_exponents(xs, ys, floor, trials=300, seed=5):
    rng = random.Random(seed)
    exponents = []
    for _ in range(trials):
        idxs = [rng.randrange(len(xs)) for _ in xs]
        bx = [xs[i] for i in idxs]
        by = [ys[i] for i in idxs]
        fit = fit_power(bx, by, floor)
        if fit and 0 < fit['exponent'] < 2:
            exponents.append(fit['exponent'])
    exponents.sort()
    lo = exponents[int(0.05 * len(exponents))]
    mid = exponents[int(0.50 * len(exponents))]
    hi = exponents[int(0.95 * len(exponents))]
    return lo, mid, hi, len(exponents)

for name, xs, ys in [('alpha', N_M, loss_by_N), ('beta', D_B, loss_by_D)]:
    lo, mid, hi, count = bootstrap_exponents(xs, ys, best['floor'])
    print(name, 'p05/median/p95=', round(lo, 3), round(mid, 3), round(hi, 3), 'valid=', count)

## コスト見積もりへつなぐ

スケーリング則で候補配分を決めても、壁時計時間、GPU 台数、稼働率、単価で実行可能性が変わります。性能予測とコスト見積もりを同じ表で見ないと、良い実験計画にはなりません。

In [ ]:
def cost_estimate(train_flops, gpu_tflops=250.0, num_gpus=64, utilization=0.35, usd_per_gpu_hour=1.8):
    seconds = train_flops / (gpu_tflops * 1e12 * num_gpus * utilization)
    hours = seconds / 3600
    usd = hours * num_gpus * usd_per_gpu_hour
    return hours, usd

for C in [1e20, 3e20, 1e21]:
    n, d = optimal_for_compute(C)
    hours, usd = cost_estimate(C)
    print(f'C={C:.1e} N={n/1e6:7.1f}M D={d/1e9:7.1f}B hours={hours:7.2f} cost_usd={usd:8.2f}')

スケーリング則は、小規模観測から大規模学習の計画を作るための経験的な道具です。床と指数を推定し、同じ計算量で N と D の配分を選び、外挿の不確かさとコストを同時に見ます。データ品質、アーキテクチャ、optimizer、トークナイザ、重複除去が変われば式の係数も変わるため、予測値は実験を止める理由ではなく、次の実験を選ぶための根拠として扱います。